In [ ]:
/*
=================================================================
       HANDLING LARGE FILES - BEGINNER NOTES (HINGLISH)
=================================================================

-----------------------------------------------------------------
SECTION 1: PROBLEM - NAIVE UPLOAD KYUN FAIL HOTA HAI?
-----------------------------------------------------------------

PARAGRAPH:
Socho aapke paas 2 GB ki video hai. Aapne puri file EK HI HTTP
request me bhejni shuru ki. 20 minute ho gaye, 80% upload ho
chuka hai... aur NETWORK DROP! Naive system me sab kuch shuru
se phir se - 0% se! Yeh all-or-nothing tarika hai. Server bhi
pareshan hota hai - 2 GB file memory me buffer ki toh RAM khatam,
100 log ek saath upload karein toh 200 GB buffering chahiye!
Load balancer aur proxy ki timeout limits lambe requests ko maar
deti hain. Asli sulah: file ko EK fragile operation ki tarah mat
dekho - use chhote-chhote INDEPENDENT pieces me todo.

DIAGRAM:

    NAIVE UPLOAD (ek hi lambi request = ek hi ande me pura bharta):

    [Client] ---(2 GB ek saath, 20 min lambi request)---> [Server] ---> [Storage]
                                    |
                          Connection DROP @ 80%
                                    |
                                    v
                       1.6 GB ka kaam VASTE! :( 
                       0% se SHURU karo phir se

    Naive ke 5 BADE PROBLEM:
    [Memory exhaustion]  - 2GB buffer = RAM gayi
    [Timeout failures]   - LB/proxy lambe requests ko maar dete hain
    [No resume]          - 99% pe drop = 0% se phir se
    [Single point fail]  - server restart = chalu upload GAYA
    [Resource blocking]  - thread/connections busy = doosre users
                           slow

BULLET POINTS:
   * Naive = ek lambi all-or-nothing request - kabhi mat karo
   * 2 GB @ 10 Mbps = ~27 minute - itni lambi request infra
     almost HAMESHA kill kar deta hai
   * Interview ka core message: file ko chhote independent
     pieces me todo, retries ko SAFE banao, bytes ko seedha
     storage bhejo, metadata alag rakho
   * Interview answer 4 steps me: clarify -> chunk/resume ->
     pre-signed URLs -> metadata DB + async processing +
     CDN downloads

-----------------------------------------------------------------
SECTION 2: PATTERN 1 - CHUNKED UPLOADS (Dabbe-dabbe me bhejo)
-----------------------------------------------------------------

PARAGRAPH:
Core idea simple hai: 2 GB file ko 64 MB ke CHHOTE chunks me todo
aur har chunk alag HTTP request me bhejo. Ek chunk fail hua toh
SIRF WO chunk dobara bhejo - puri file nahi! Jaise ghar shifting
me saaman ko dabbon me pack karna - ek dabba toota toh baaki saare
salammat. 2 GB = 32 chunks (64 MB wale). Ab ek operation me ghanton
ka risk nahi, balki kai chhote low-risk operations hain.

DIAGRAM:

    [Client: 1 GB file]
        |
        | todo 64-64 MB ke tukdo me
        v
    [C1][C2][C3][C4].......[C16]
        |    |    |    |
        +----+----+----+--> har chunk ALAG HTTP request
                             (parallel bhi bhej sakte ho!)
                             |
                             v
                        [SERVER / STORAGE]

    3 PHASES (yeh flow ratta maar lo):
    PHASE 1 - INIT:
       POST /uploads/init {file_name, file_size, checksum}
       Server: upload session banao, chunk size batao
       Response: {upload_id, chunk_size: 64MB, total_chunks: 16}

    PHASE 2 - CHUNK UPLOADS:
       PUT /uploads/{id}/chunks/0  + X-Chunk-SHA256 header
       Server: checksum verify -> {received: true}

    PHASE 3 - COMPLETE:
       POST /uploads/{id}/complete
       Server: saare chunks verify -> final object assemble ->
       metadata DB me file record banao
       Response: {file_id, status, url}

BULLET POINTS:
   * Chunk size aam taur pe 4-64 MB hota hai
   * Har chunk = independent request = fail hone pe SIRF wahi
     chunk retry
   * Parallel upload possible hai (concurrency limit ke saath)
   * S3 ki limits yaad rakho: part min 5 MB, max 10,000 parts,
     object max 5 TB

-----------------------------------------------------------------
SECTION 3: RESUMABLE UPLOADS (Wahin se shuru karo jahan ruke the)
-----------------------------------------------------------------

PARAGRAPH:
Connection drop hua? Client ko andaza lagane ki zaroorat NAHI ki
kahan ruka tha. Wo server se poochta hai: "bhai, kaunse chunks
mile?" GET /status call se server batata hai: chunks 0-4 mile,
5-15 missing. Client sirf chunk 5 se resume karta hai. 90% chadh
chuki file sirf 10% dobara bhejti hai. Naive approach me wahi
failure matlab 100% dobara. WhatsApp pe bada video bhejne se
pehle network chala gaya - resume hua toh khushi, shuru se hua
toh frustration - wahi kahani hai!

DIAGRAM:

    16 chunks upload karte karte network DROP (chunk 5 pe)

    CLIENT                                SERVER
      |                                      |
      |--- GET /uploads/{id}/status -------->|
      |                                      |
      |<-- {received: [0,1,2,3,4],           |
      |     missing: [5..15]} ---------------|
      |                                      |
      |  (achha! sirf 5-15 bhejne hain)      |
      |--- PUT chunks/5 -------------------> |  <-- RESUME!
      |--- PUT chunks/6 -------------------> |
      |--- ...chunks 15 tak...               |
      |--- POST /complete -----------------> |  --> DONE! :)

BULLET POINTS:
   * Resume = server hi SACH batata hai kaunse chunks aaye
   * Client guess NAHI karta, STATUS QUERY karta hai
   * 90% complete file fail hui = sirf 10% ka dobara cost
   * Yehi cheez mobile networks (unstable) me GAME-CHANGER hai

-----------------------------------------------------------------
SECTION 4: IDEMPOTENT CHUNK UPLOADS (Retry SAFE kaise ho)
-----------------------------------------------------------------

PARAGRAPH:
Ek subtle problem: client ne chunk 5 bheja, server ne store kar
liya, PAR response network me kho gaya! Client ko laga fail hua,
wo dobara bhejega. Dobara bhejna waste hai, par "duplicate"
banna aur bhi bura. Solution: har chunk ko ADDRESSABLE banao.
Chunk ki identity = (upload_id, chunk_index), verification =
checksum. Retry pe rules: same upload_id + same index + same
checksum = "already hai bhai, success" (naya store nahi karna).
Checksum ALAG aaya = client galat byte range bhej raha hai =
REJECT! Checksum integrity deta hai, upload_id + index
idempotency dete hain.

DIAGRAM:

    RETRY SCENARIO:

    Attempt 1: PUT chunks/5  --> server STORE karta hai
               <-- response LOST (network!)
    Attempt 2: PUT chunks/5  --> server CHECK karta hai:
               |
               +-- upload_id same? YES
               +-- chunk_index same? YES  
               +-- checksum same? YES --> "received: true"
                   (dubara store NAHI, bas success bata do!)

    AGAR CHECKSUM ALAG HO?
    --> REJECT! (client galat data overwrite karne ki koshish
        kar raha hai - corruption ruk gayi!)

BULLET POINTS:
   * Idempotency = same request kitni baar bhejo, result SAME
   * Identity = (upload_id, chunk_index), Proof = checksum
   * Same + same + same = success (duplicate nahi banega)
   * Same + same + different = reject (corruption nahi hogi)
   * Isse retries SAFE ho jate hain - na duplicates, na corrupt
     final file

-----------------------------------------------------------------
SECTION 5: CHUNK SIZE KAISE CHOOSE KAREIN (Trade-off ka khel)
-----------------------------------------------------------------

PARAGRAPH:
Chunk size ek balance hai. CHHOTE chunks (1-4 MB) = resume fine
grained (kam data waste) par zyada HTTP requests = overhead.
BADE chunks (100+ MB) = kam requests par fail hone pe bahut data
dobara bhejna padega. Rule of thumb: MOBILE clients (unstable
network) ke liye 4-8 MB; DESKTOP/server (stable network) ke liye
16-64 MB. Best of all: ADAPTIVE sizing - shuru me chhote chunks,
agar upload smooth chal raha hai toh size badhate jao!

DIAGRAM:

    CHHOTA CHUNK (1-4 MB)          BADA CHUNK (100+ MB)
    [10][10][10][10][10]...        [=========100=========]
     + resume me kam waste          + kam HTTP requests
     + mobile ke liye best          + stable network ke liye best
     - zyada requests/overhead      - fail hua toh BARBAARI!

              ADAPTIVE (best!):
    [5][5][5] ---> sab sahi chal raha ---> [10][20][40]...
    (shuru me chhote, phir badhte jao!)

BULLET POINTS:
   * Mobile/unstable: 4-8 MB | Desktop/stable: 16-64 MB
   * Chhota = fine resume + low memory + zyada overhead
   * Bada = kam requests + mehenga retry + zyada memory
   * Adaptive sizing = start small, grow on success
   * Provider limits yaad rakho (S3: min part 5 MB, max 10k
     parts, max 5 TB object)

-----------------------------------------------------------------
SECTION 6: PATTERN 2 - PRE-SIGNED URLS (Bytes ko seedha storage bhejo)
-----------------------------------------------------------------

PARAGRAPH:
Chunked uploads ne resume solve kiya, par agar server ke PAAS se
data jata hai toh problem hai: 2 GB ka data aapke app servers pe
aayega - bandwidth, memory, CPU sab khayega, aur double bandwidth
charge lagega (ek baar receive, ek baar storage forward).
SOLUTION: client ko storage (S3/GCS/Azure) se SEEDHA baat karne
do! Server sirf AUTH aur METADATA handle kare. Server ek PRE-SIGNED
URL deta hai - ek URL jisme AUTHORIZATION DHAKI HUI hai. Jaise
hotel ka keycard - aapke paas hotel ki master key nahi hai (real
credentials), bas ek limited-time, limited-purpose chaabi hai!

DIAGRAM:

    BINA PRE-SIGNED (sab kuch server ke through = BOTTLENECK):
    [Client] --2 GB--> [App Server] --2 GB--> [Storage]
                        (dabba gaya, paisa double gaya)

    PRE-SIGNED URL KE SAATH (direct path!):
    [Client] --metadata--> [API Server]
                |
                |<-- pre-signed URLs (auth embedded) --|
                |
    [Client] ====== 2 GB SEEDHA ======> [S3 Storage]
                |                          |
                +---- POST /complete ----> [API Server]
                      (verify + file record banao)

    PRE-SIGNED URL ke andar kya-sign hua hai:
    https://storage.../file.mp4 ?X-Signature=abc123
                                &X-Expires=1702900000
                                &X-Algorithm=AWS4-HMAC-SHA256
    (kaunsa OPERATION + kaunsa OBJECT + kab tak VALID)

BULLET POINTS:
   * Server credentials client ko KABHI nahi deta - sirf
     signed, EXPIRING, scoped URL deta hai
   * URL mein locked hota hai: operation (PUT/GET), object path,
     expiration (1-2 hrs upload, 15-60 min download)
   * Fayde: server ka load DOWN, bandwidth cost DOWN, clients
     geographically-paas storage nodes pe fast, storage alag
     scale hota hai
   * Security: URL kisi ke paas hai toh wo use kar sakta hai
     (password-reset link jaisa model) - isliye SHORT expiry,
     content-type/size conditions sign karo, IP restrictions
     (possible ho toh), usage monitor karo
   * NOTE: S3 pe har chunk ko alag OBJECT banakar bhejoge toh
     completion pe jodna mehenga padega (UploadPartCopy). Provider
     ka NATIVE multipart API use karo - parts sign karo ek hi
     object ke!

-----------------------------------------------------------------
SECTION 7: PATTERN 3 - S3 MULTIPART UPLOAD (Provider ka built-in)
-----------------------------------------------------------------

PARAGRAPH:
Agar cloud storage use kar rahe ho (S3/GCS/Azure), toh chunk
assembly KHUD mat banao - provider ne pehle se bana rakha hai!
S3 Multipart Upload = initiate (upload_id lo) -> parts upload
karo (5 MB - 5 GB each, max 10,000 parts) -> complete (S3 khud
parts ko jodkar final object banata hai) -> ya abort. Har part
upload pe server ETag return karta hai (proof). Parts PARALLEL
upload ho sakte hain - 4 threads, 16 parts baant ke!

DIAGRAM:

    S3 MULTIPART FLOW:

    [1] POST /bucket/object?uploads
        <-- <UploadId>xyz789</UploadId>

    [2] PUT /bucket/object?partNumber=1&uploadId=xyz789  [data]
        <-- ETag: "abc123"   (har part ka proof milta hai)
        (parts 2..N bhi aise hi...)

    [3] POST /bucket/object?uploadId=xyz789  + part list + ETags
        <-- S3 saare parts jodke FINAL object banata hai!

    [4] DELETE /bucket/object?uploadId=xyz789   (abort/cancel)

    PARALLEL UPLOAD (4 threads, speed boost):
    Thread 1: parts 1,5,9,13   |
    Thread 2: parts 2,6,10,14  |----> [S3]
    Thread 3: parts 3,7,11,15  |
    Thread 4: parts 4,8,12,16  |

BULLET POINTS:
   * Parts: 5 MB to 5 GB, max 10,000 parts, object max 5 TB
   * Parallelism se 4x speed TAB TAK jab tak ek connection
     bandwidth bhar na pa rahi ho - aapki uplink limit se
     UPAR nahi ja sakta!
   * Practice me 4-8 parallel streams kaafi hain, usse zyada
     se fayda nahi (bandwidth/disk limits aa jaati hain)
   * CHHUPA HUA KHARCHA: adhoore uploads storage KHAATE rehte
     hain aur normal listing me nahi dikhte!
   * CLEANUP: S3 Lifecycle Rule lagao - "AbortIncompleteMultipart
     Uploads" 7 din baad. Ya DB me active uploads track karke
     background job se purane abort karo

-----------------------------------------------------------------
SECTION 8: PATTERN 4 - STREAMING UPLOADS (Jab size pehle NAHI pata)
-----------------------------------------------------------------

PARAGRAPH:
Ab tak maan rahe the file size pehle se pata hai. Par LIVE video,
continuous logs, on-the-fly compression - inka size KAISE pata
karein? Streaming upload: Content-Length ki zaroorat NAHI. Data
jaise-jaise banta hai, waise-waise bhej do! HTTP/1.1 ka CHUNKED
TRANSFER ENCODING ye karta hai - har tukde ke aage uska size
(hex me), aur zero-length chunk = "khatam!" DHYAN: yeh HTTP ka
wire-format hai - application-level upload chunks (Pattern 1) se
ALAG cheez hai! Server stream ko as-it-is storage me append karta
hai, memory me kuch nahi rakhta.

DIAGRAM:

    HTTP CHUNKED TRANSFER ENCODING (wire format):

    POST /upload
    Transfer-Encoding: chunked

    5\r\n          <-- size hex me (5 bytes)
    Hello\r\n      <-- data
    6\r\n
     World\r\n
    0\r\n          <-- ZERO = "bas, khatam!"
    \r\n

    SERVER SIDE (stream karke append karo, buffer nahi):
    async for chunk in request.stream():
        append_to_object(upload_id, offset, chunk)
        offset += len(chunk)

BULLET POINTS:
   * USE karo: live video/audio, continuous logs, on-the-fly
     compress/encrypt ho raha data
   * LIMITATIONS (important!):
     - Resumability NAHI hai by default (session/part/resume
       protocol nahi hai - khud banana pade toh wapas Pattern 1!)
     - Size/checksum END me hi verify ho sakta hai
     - Kuch proxies/LBs chunked requests ko BUFFER kar dete
       hain - purpose khatam!
   * RULE: size pata ho toh explicit chunking + resume use karo.
     Streaming SIRF jab data ki nature maange

-----------------------------------------------------------------
SECTION 9: UPLOAD KE BAAD - RELIABLE COMPLETION + PROCESSING
-----------------------------------------------------------------

PARAGRAPH:
Bytes storage me pahunch gaya - kaam adhura hai! Pehla sawal:
backend ko RELIABLY kaise pata chale ki upload complete hua?
Client ka POST /complete call kamzor guarantee hai - client crash
ho sakta hai, network ja sakta hai. Aise uploads = ORPHANED bytes
(jinka koi record nahi). SOLUTION: STORAGE EVENTS! Object write
hote hi storage khud event chhodta hai (S3: ObjectCreated ->
SQS/SNS/Lambda). Backend event sunkar KHUD file ready mark karta
hai - client pe depend nahi! Doosra sawal: file aane ke baad?
Video ko transcode karna hai, thumbnail banana hai, virus scan
karna hai - ye SAB BHAARI kaam upload request me NAHI, QUEUE se
ASYNC workers me hoga! Isliye file records me STATUS hota hai:
uploaded -> processing -> ready.

DIAGRAM:

    RELIABLE COMPLETION (dual path - belt + suspenders):

    [Client] --PUT bytes--> [OBJECT STORAGE]
        |                        |
        |                        | ObjectCreated EVENT!
        | (optional:             v
        |  POST /complete)   [EVENT QUEUE] --> [FINALIZER WORKER]
        |                                            |
        |                                            v
        +---------------------------------> [METADATA DB]
                                     (verify size+checksum,
                                      phir "ready" mark karo)

    ASYNC PROCESSING PIPELINE (bhaari kaam queue me!):

    [Upload finalized] --> [QUEUE] --> [Virus scan]
                                  |--> [Transcode/thumbnail]
                                  |--> [Index for search]
                                  |--> [Status update + outputs]
    (file "ready" hoke hi doosre users ko dikhegi; virus
     fail hua toh QUARANTINE!)

BULLET POINTS:
   * Client complete call = FAST interactive response
   * Storage event = BACKSTOP (client confirm na kare toh bhi
     file finalize hogi) - dono saath use karo
   * "Ready" sirf tab jab backend ne verify kiya: object exist +
     size + checksum match
   * Bhaari kaam (transcode, scan, index) upload request se
     BAHAR - queue + worker pool me async
   * STATUS FIELD: uploaded / processing / ready - interview me
     ye bolna dikhata hai ki aap poora LIFECYCLE soch rahe ho
   * Virus scan fail = file QUARANTINE, kabhi "ready" nahi

-----------------------------------------------------------------
SECTION 10: DOWNLOAD OPTIMIZATIONS - RANGE REQUESTS (206)
-----------------------------------------------------------------

PARAGRAPH:
Downloads ki apni problem hai - naive download bhi wahi all-or-
nothing hai. HTTP ka built-in ilaaj: RANGE REQUESTS! Puri file
mat mango, sirf BYTE RANGE mango. "Range: bytes=0-1048575" =
pehla 1 MB do. Server 206 Partial Content ke saath Content-Range
header deta hai. Isse 3 superpowers milti hain: (1) RESUMABLE
downloads - 50 MB mila tha, wahi se resume: Range: bytes=
52428800-. (2) PARALLEL downloads - 4 connections, 4 ranges,
ek saath! (3) VIDEO SEEKING - user 1:30:00 pe jump kare toh
player SIRF wahan ka byte-range mangta hai, shuru se sab nahi!

DIAGRAM:

    RANGE REQUEST:

    GET /files/video.mp4
    Range: bytes=0-1048575         (sirf pehla 1 MB!)
         |
         v
    HTTP/1.1 206 Partial Content
    Content-Range: bytes 0-1048575/104857600
         |
    [pehla 1 MB data]

    PARALLEL DOWNLOAD (4 threads, phir client jodta hai):
    Thread 1: bytes 0-25M    |
    Thread 2: bytes 25-50M   |----> client REASSEMBLE
    Thread 3: bytes 50-75M   |      karta hai
    Thread 4: bytes 75-100M  |

BULLET POINTS:
   * 206 Partial Content = "sirf hissa bhej raha hoon"
   * Content-Range header = kaunse bytes + total size batata hai
   * Resume: Range: bytes=52428800- (jahan ruka tha wahi se)
   * Video seeking ke liye container index/segment manifest se
     timestamp -> byte range map hota hai
   * Parallel downloads high-latency links pe single connection
     se behtar bandwidth bhar paate hain

-----------------------------------------------------------------
SECTION 11: CDN + PRIVATE FILES (Speed + Security)
-----------------------------------------------------------------

PARAGRAPH:
Popular file ko har baar origin se laana = slow + origin pe bojh.
CDN edge servers duniya bhar me file CACHE kar lete hain - pehla
user region me origin se laata hai, baaki sab paas wale edge se!
NYC user ko NYC edge se, Tokyo user ko Tokyo edge se - fast!
Cache headers lagana mat bhoolna: Cache-Control + ETag + sabse
important Accept-Ranges: bytes (isise CDN ke through range
requests/resume/seek chalta hai!). PRIVATE files ke liye: bytes
apne server se proxy mat karo - wahi direct-path idea ULTA use
karo! Server sirf AUTH check kare aur SHORT-LIVED signed URL
(15-60 min) return kare - client storage/CDN se seedha download
kare.

DIAGRAM:

    CDN DISTRIBUTION:

    [Origin Storage]
        ^  (sirf PEHLI baar - cache miss)
        |
    [Edge NYC]  [Edge London]  [Edge Tokyo]
        ^            ^             ^
    NYC users    London users  Tokyo users
    (aage se sab CACHE se - FAST!)

    CDN-friendly headers:
    Cache-Control: public, max-age=31536000, immutable
    ETag: "abc123"          <- immutable + versioned names
    Accept-Ranges: bytes    <- range/resume/seek ke liye ZARURI

    PRIVATE FILE DOWNLOAD (signed URL):

    [Client] --"file do"--> [API Server: auth check]
                                 |
                    <-- signed GET URL (15-60 min expiry)
                                 |
    [Client] =====download===== [Storage/CDN]
    (server bytes ko KABHI touch nahi karta, sirf
     ALLOW/DENY decide karta hai!)

BULLET POINTS:
   * CDN fayde: low latency, high throughput, origin ka load
     kam, geographic redundancy
   * Accept-Ranges: bytes header = range requests chalengi
   * Long max-age SIRF immutable/versioned names ke liye
   * Private files: public cache NAHI - signed CDN URLs ya
     signed cookies
   * Signed URL = scoped (ek object) + expiring - share karo
     toh bhi jaldi mar jayega, doosre objects nahi kholega
   * Uploads wala principle ulta: authorize karne wala,
     bytes le jaane wala alag!

-----------------------------------------------------------------
SECTION 12: STORAGE ARCHITECTURE - Kahan rakhein, kaise rakhein
-----------------------------------------------------------------

PARAGRAPH:
File aane ke baad KAHAN rahegi? 3 options: OBJECT STORAGE (S3/
GCS) - bade immutable files ke liye BEST, web apps ka default;
DISTRIBUTED FILE SYSTEM (HDFS) - analytics ke liye, web serving
ke liye nahi; DATABASE BLOB columns - sirf 1 MB se chhoti files
ke liye, badi files pe performance barbaad. Sabse badi galti:
metadata (naam, size, owner, permissions) aur DATA (asli bytes)
ko ek jagah treat karna. INKO ALAG KARO! Metadata = PostgreSQL
me (queries fast, transactions), Bytes = object storage me
(bade objects, CDN integration). DB me bas ek mapping:
file_id -> object_key.

DIAGRAM:

    STORAGE KAUNSA?
    +----------------------------+---------------------------+
    | Object Storage (S3/GCS)    | badi immutable files -    |
    |                            | BEST CHOICE (default)     |
    | Distributed FS (HDFS)      | analytics; small files    |
    |                            | ke liye AVOID             |
    | DB BLOB columns            | sirf <1 MB files; badi    |
    |                            | files pe DEGRADE hota hai |
    +----------------------------+---------------------------+

    METADATA vs DATA SEPARATION:

    [Client] --> [API Server] --file_id lookup--> [Metadata DB]
                          |                        (PostgreSQL:
                          |                         naam, owner,
                          |                         permissions,
                          |                         object_key)
                          |
                          +----object_key se------> [OBJECT STORAGE]
                                                    (asli bytes,
                                                     S3/GCS)

BULLET POINTS:
   * Web apps ka default = object storage
   * Metadata alag = queries fast (user ki files dhundhna =
     sirf DB, object storage ko touch nahi)
   * Metadata DB = source of truth (transactions, permissions,
     uniqueness yahan manage hota hai)
   * Alag retention: metadata hamesha (audit), files bade
     retention period baad delete
   * Har layer apne hisab se scale hoti hai

-----------------------------------------------------------------
SECTION 13: CONTENT-ADDRESSABLE STORAGE (Content hi address hai)
-----------------------------------------------------------------

PARAGRAPH:
File ko random path pe store karne ki jagah - file ke CONTENT ka
HASH nikalo aur WAHI uska address banao! Path jaise:
blobs/ab/12/sha256hash... (pehle 2 chars folders me - ek folder
me crore files na bhar jayein). Fayde: (1) DEDUPLICATION - do
users ne same file upload ki = same hash = bytes EK HI BAAR store,
dono metadata records usi pe point karenge! (2) IMMUTABILITY -
content badla toh hash badal gaya - integrity check trivial.
(3) CONCURRENT uploads safe - conditional writes (copy_if_absent)
se race jeet-ti hai ek hi write. PAR ek tension hai: content
address ke liye hash chahiye, par pre-signed direct upload me
bytes server ko dikhte hi NAHI! Solution: (a) client hash bheje
(par blind trust nahi - server baad me verify kare), YA (b)
best: pehle TEMP location pe direct upload, phir storage EVENT
se background job hash nikal ke content-addressed layout me
move kare - yehi processing pipeline jaisa pattern hai!

DIAGRAM:

    CONTENT-ADDRESSABLE STORAGE:

    file content --> SHA256 --> hash: "ab12cd34..."
                                    |
    Path: blobs/ab/12/ab12cd34...  <-- hash hi KEY hai!
          (prefix folders = ek folder me kam files)

    DEDUP ka magic:
    User A uploads "report.pdf"  --> hash X --> store (first time)
    User B uploads same file     --> hash X --> ALREADY HAI!
                                             metadata point -> X
                                             (bytes ek bhi naye nahi!)

    DIRECT UPLOAD + HASHING ka jhagda (aur hal):
    Problem: server ne bytes dekhi hi nahi, hash kaise?
    Hal: temp upload -> storage EVENT -> worker hash nikalta
         hai -> content-addressed location pe move/map
         (bytes server-side EK baar padhe jaate hain)

BULLET POINTS:
   * Hash = address = same content hamesha same jagah
   * Whole-file dedup free me milta hai (popular PDFs/media ke
     liye badi bachat)
   * Integrity = dobara hash karo, match? file intact!
   * Concurrent writes = conditional writes / idempotent store
   * Server-side hash POST-upload event me karo - direct path
     + dedup dono milte hain
   * Metadata DB: user-file paths -> content hashes (kai files
     -> same hash = dedup)

-----------------------------------------------------------------
SECTION 14: DEDUP ADVANCED - BLOCK LEVEL + CDC
-----------------------------------------------------------------

PARAGRAPH:
Poori file same ho toh hash-dedup kaam karta hai. Par agar file
99% same ho, bas 1 paragraph badla ho? Poori file dobara store?
BLOCK-LEVEL DEDUP: file ko blocks me todo, har block ka hash,
sirf NAYE blocks store karo. File B ko File A ke 2 blocks mile -
sirf ek naya block store hua! PAR fixed-size blocks me BUG hai:
file ke SHURU me ek character insert karo - SABKE block
boundaries shift! Dedup barbaad. SOLUTION: CONTENT-DEFINED
CHUNKING (CDC) - boundaries content khud decide karta hai
(rolling hash jaise Rabin fingerprint se pattern dhundo, wahan
kaato). Ab insert ke baad bhi purane chunks SALAMMAT rehte hain!
Ye backup/sync systems (Dropbox) ki advanced cheez hai - basic
interview me zaroori nahi jab tak sync/storage-efficiency core
requirement na ho.

DIAGRAM:

    BLOCK-LEVEL DEDUP:

    File A: [Block1: abc][Block2: def][Block3: ghi]
    File B: [Block1: abc][Block2: xyz][Block3: ghi]
                              |
                    sirf Block2 NAYA hai!
    Storage: [abc][def][ghi][xyz]  <-- 4 blocks, 6 nahi!

    FIXED CHUNKING ka PROBLEM (1 letter insert = sab shift):

    PEHLE:   [AAAA]  [BBBB]  [CCCC]
    'X' insert shuru me:
    BAAD ME: [XAAA]  [ABBB]  [BCCC]   <-- TEENO badal gaye! :(

    CDC ka JADOO (boundaries content se):
    PEHLE:   [AAA]  [BBBBB]  [CC]
    'X' insert shuru me:
    BAAD ME: [X]  [AAA]  [BBBBB]  [CC]  <-- purane chunks SALAMMAT! :)

BULLET POINTS:
   * Block dedup = file 99% same ho toh sirf 1% ka naya storage
   * Backup/sync systems isi se sirf CHANGED blocks upload karte
   * Fixed blocks ki kamzori: insert = boundaries shift = dedup
     kharab
   * CDC = content ke pattern pe kaato (rolling hash) - boundaries
     STABLE rehte hain
   * Compression: text/logs/WAV/BMP compress karo; JPEG/MP4/ZIP
     ko MAT karo (pehle se compressed hain - ulta bade ho sakte!)

-----------------------------------------------------------------
SECTION 15: MASTER FLOW (Sab kuch ek saath - 2 GB upload)
-----------------------------------------------------------------

    [USER: 2 GB file select ki]
        |
        |  file ko 64MB ke 32 chunks me toda
        v
    POST /uploads/init {name, size, checksum}
        |
        v
    [API: session banao + PRE-SIGNED URLs banao]
        |                        {upload_id, chunk_urls[]}
        v
    [Client: PARALLEL chunks upload] =====direct=====> [S3]
        |           (4-8 streams)
    Connection DROP @ chunk 20! 
        |
        v
    GET /uploads/{id}/status
        <-- {received: [0-19], missing: [20-31]}
        |
        v
    RESUME: sirf chunks 20-31 dobara =====> [S3]
        |
        v
    POST /uploads/{id}/complete
        |                    (fast response)
        +---------------------------> [S3: ObjectCreated EVENT]
        |                                       |
        v                                       v
    {file_id, url}                    [QUEUE --> Finalizer Worker]
        |                             verify size+checksum -> READY
        v                                       |
    [ASYNC PIPELINE]                            |
    virus scan -> transcode -> index            |
        |                                       v
        v                              [METADATA DB]
    status: "ready"! <-- ab file doosre users ko dikhegi

    DOWNLOAD side:
    [User] -> [CDN edge] --cache hit--> fast!
                   |cache miss
                   v
              [Origin S3] --Range requests (206)--> resume/seek
    private file? -> API auth -> short-lived signed URL -> direct

BULLET POINTS (sab patterns kaise JUDTE hain):
   * Pre-signed URLs = bytes DIRECT storage ko (server bypass)
   * Parallel chunks = throughput (jab 1 connection kam pade)
   * Status query = RESUME (drop hone pe kaunse chunks gaye?)
   * Content-addressing = DEDUP (same file = ek hi object)
   * Metadata DB alag = queries + transactions saaf
   * Storage events + async pipeline = client confirm na kare
     toh bhi finalize + bhaari kaam background me
   * Range requests + CDN = downloads bhi resumable + fast

-----------------------------------------------------------------
SECTION 16: INTERVIEW CHECKLIST (Ek nazar me)
-----------------------------------------------------------------

    UPLOAD PATH:
    * Chunked uploads (configurable size) ....... resume+parallel
    * Progress tracking + resume ................ user ka kaam safe
    * Pre-signed URLs ........................... server load down
    * Chunk index + checksum .................... safe retries
    * Session timeout + cleanup ................. orphan data nahi
    * Storage events for completion ............. client crash-proof
    * Async pipeline (scan/transcode/index) ..... user ka wait nahi

    DOWNLOAD PATH:
    * Range requests (206) ...................... resume + seek
    * CDN ....................................... latency + origin load
    * Cache headers (Accept-Ranges!) ............ CDN-friendly
    * Signed URLs (private) ..................... access control
      bina bytes proxy kiye

    STORAGE + RELIABILITY:
    * Object storage (bytes) + Metadata DB ...... alag-alag scale
    * Content-addressable storage ............... whole-file dedup
    * Idempotent chunks ......................... retry = safe
    * Garbage collection ........................ storage leak nahi
    * End-to-end checksum ....................... integrity proof

-----------------------------------------------------------------
QUICK REVISION (Interview one-liners - yaad kar lo!)
-----------------------------------------------------------------
   * Naive upload = all-or-nothing - memory khaata, timeout
     maarta, resume nahi hota
   * Chunked upload = chhote independent pieces; fail pe SIRF
     missing chunk retry (init -> chunks -> complete)
   * Status GET = client guess nahi karta, server sach batata hai
   * Idempotency = (upload_id, chunk_index) + checksum - retry
     safe, duplicate/corruption zero
   * Chunk size: mobile 4-8 MB | desktop 16-64 MB | ADAPTIVE best
   * Pre-signed URL = embedded authorization + expiry - bytes
     seedha storage ko, server sirf metadata+auth
   * S3 multipart: initiate -> parts (5MB-5GB, max 10k) ->
     complete; 7-day lifecycle rule se adhoore uploads saaf
   * Streaming = jab size pehle nahi pata (live video/logs) -
     resumability ka fayda nahi
   * Client "complete" call kamzor hai - STORAGE EVENTS (Object
     Created) se backend KHUD finalize kare
   * File status: uploaded -> processing -> ready (scan/
     transcode/index sab ASYNC queue me)
   * Downloads: Range: bytes=0-1048575 -> 206 Partial Content
     (resume + parallel + video seek)
   * CDN + Accept-Ranges: bytes = fast + resumable downloads
   * Private file = signed URL (15-60 min) - server bytes ko
     touch nahi karta
   * Metadata (PostgreSQL) alag, bytes (S3) alag - file_id ->
     object_key mapping
   * Content-addressable storage = hash hi address = free dedup
     + immutability
   * CDC (content-defined chunking) = boundaries content se -
     insert ke baad bhi chunks bachte hain (Dropbox-style sync)
=================================================================
*/